# TP 2 - Assistant RAG amélioré

Ce notebook compare trois stratégies de recherche avancées sur la base vectorielle V2 : **Multi-Query**, **HyDE** et **Reranking**.
La requête et le prompt système sont identiques dans les trois méthodes pour permettre une comparaison directe sans biais.

### 0.1. Objectif
- **TP 2_1** : Préparer la base vectorielle V1 (chunking par caractères)
- **TP 2_2** : Créer un assistant RAG simple
- **TP 2_3** : Préparer la base vectorielle V2 (chunking par en-têtes Markdown)
- **TP 2_4** : Comparer trois stratégies de recherche avancées : Multi-Query (reformulations), HyDE (document hypothétique), Reranking (reclassement LLM)

### 0.2. Documentation générale

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

[ChromaDB](https://docs.trychroma.com/)

In [ ]:
import json

from pydantic import BaseModel

from shared.config import ROOT_DIR
from shared.rag_utils import (
    RAGAssistant,
    RAGChunk,
    rag_deduplicate_and_sort_chunks,
    rag_embed_text_batch,
    rag_embed_text_batch_local,
)

# INFO : Choix entre local ou cloud (LLM)
from shared.llm_utils import (
    LLMRequest,
    run_llm, # Cloud
    #run_llm_local as run_llm, # Local
    run_llm_structured, # Cloud
    #run_llm_local_structured as run_llm_structured, # Local
)

# INFO : Choix entre local ou cloud (embeddings)
rag_embed_fn = rag_embed_text_batch # Cloud
#rag_embed_fn = rag_embed_text_batch_local # Local

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
VECTOR_DB_DIR = DATA_DIR / "chroma_db_rag_v2"

MULTI_QUERY_COUNT = 3
MULTI_QUERY_TOP_K = 10
HYDE_TOP_K = 10
RERANK_FINAL_K = 10

### 0.3. Initialiser l'assistant RAG

In [ ]:
rag_assistant = RAGAssistant(persist_dir=VECTOR_DB_DIR, top_k=10, embed_fn=rag_embed_fn)

### 0.4. Récapitulatif des fonctions utilisées dans ce notebook

**Fournies**

- `run_llm` : envoie une requête en texte libre et retourne un `LLMResponse`
- `run_llm_structured` : comme `run_llm`, mais impose une sortie JSON conforme à un `response_schema`
- `LLMRequest` : classe qui représente les données d'entrée d'un appel LLM
- `LLMResponse` : classe qui représente les données de sortie utiles (texte final, tokens, données brutes)

--> Disponibles dans `shared/llm_utils.py`

- `RAGAssistant` : classe qui implémente la recherche vectorielle sur base Chroma persistée
- `rag_deduplicate_and_sort_chunks` : fonction qui déduplique et trie une liste de chunks par score décroissant

--> Disponibles dans `shared/rag_utils.py`

### 0.5. Use case principal

La requête et le prompt système sont communs aux 3 méthodes.
Ne pas les modifier entre sections : sinon la comparaison devient invalide.

In [ ]:
user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

system_prompt = """
Tu es un assistant de planification de voyage basé sur la méthode RAG.

Règles :
- Utiliser uniquement les faits présents dans CONTEXTE.
- Ne jamais inventer prix, dates, horaires, adresses ou transports.
- Si une information manque, écrire: "Je ne sais pas à partir du contexte fourni."
- Citer les faits au format [source - chunk id].

Style attendu :
- Être concis et pratique.
- Respecter les contraintes de durée et de budget.
- Utiliser cette structure: Résumé, Itinéraire, Budget, Informations manquantes.
"""

---
## 1. Recherche Multi-Query

Une seule formulation peut manquer des passages pertinents si le vocabulaire diffère de celui des documents.

Multi-Query génère **plusieurs reformulations** de la requête : chacune est envoyée séparément au vecteur store, et les résultats sont fusionnés. La **déduplication** garantit qu'un même chunk ne remonte pas plusieurs fois.

Comme en TP1, on utilise `run_llm_structured` (sortie structurée) pour garantir une liste de requêtes en JSON valide, plutôt que de demander un JSON en texte libre et de le parser à la main.

### 1.1. Définir le schéma et le prompt de reformulation

In [ ]:
class MultiQueryReformulations(BaseModel):
    queries: list[str]


multi_query_system_prompt = """
Tu réécris des requêtes pour la recherche RAG.
Génère des requêtes variées qui conservent exactement l'intention utilisateur.
Règles :

(1) Interpréter la demande utilisateur
- Prendre en compte les préférences (ex: peu touristique, végétarien, etc.)
- Retirer les contraintes qui seront traitées par la récupération de chunks (ex: dates)

(2) Générer des requêtes complémentaires
- Utiliser des reformulations et synonymes réellement différents
- Formuler chaque requête comme du texte qu'on peut retrouver dans des chunks de guide
"""

multi_query_user_prompt = (
    f"Demande utilisateur :\n{user_query}\n\n"
    f"Génère exactement {MULTI_QUERY_COUNT} reformulations."
)

multi_query_result = await run_llm_structured(
    LLMRequest(system_prompt=multi_query_system_prompt, user_prompt=multi_query_user_prompt),
    response_schema=MultiQueryReformulations,
)

### 1.2. Parser et inspecter les requêtes générées

Vérifier que les reformulations sont vraiment différentes.
Concrètement, chaque requête doit apporter un vocabulaire ou un angle nouveau; sinon elle coûte des appels API sans gain.

In [ ]:
generated_queries = json.loads(multi_query_result.output)["queries"]

print(f"Requête utilisateur d'origine :\n  {user_query.strip()}\n")
print(f"{len(generated_queries)} requêtes de recherche générées :")
for i, q in enumerate(generated_queries, start=1):
    print(f"  Q{i}: {q}")

### 1.3. Récupérer les chunks pour chaque requête

Chaque requête est envoyée séparément au vecteur store.
Ensuite, fusionner les résultats et retirer les doublons pour ne garder qu'une occurrence par chunk.

In [ ]:
multi_query_chunks: list[tuple[RAGChunk, float]] = []
for generated_query in generated_queries:
    multi_query_chunks.extend(rag_assistant.search(query=str(generated_query), top_k=MULTI_QUERY_TOP_K))

### 1.4. Dédupliquer et trier les chunks

In [ ]:
multi_query_final_chunks = rag_deduplicate_and_sort_chunks(multi_query_chunks)[:RERANK_FINAL_K]

### 1.5. Inspecter les chunks récupérés

Contrôle concret : vérifier manuellement que les chunks retenus répondent bien à la demande.

In [ ]:
print(f"Total de chunks récupérés : {len(multi_query_chunks)}")
print(f"Après déduplication     : {len(multi_query_final_chunks)}")
print(f"Fichiers et volume :")
source_counts = {}
for chunk, _score in multi_query_final_chunks:
    source = chunk.source
    if source in source_counts:
        source_counts[source] += 1
    else:
        source_counts[source] = 1
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")
print("\n"*5)

for rank, (chunk, score) in enumerate(multi_query_final_chunks, start=1):
    print(f"#{rank} | score={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(chunk.text)
    print("\n"*5)

### 1.6. Générer la réponse ancrée

In [ ]:
multi_query_context = "\n\n".join(
    [f"[{chunk.source} - chunk {chunk.chunk_id}]\n{chunk.text}" for chunk, _score in multi_query_final_chunks]
)
multi_query_grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{multi_query_context}"

multi_query_run_result = await run_llm(
    LLMRequest(system_prompt=multi_query_grounded_system_prompt, user_prompt=user_query)
)
multi_query_answer = multi_query_run_result.output

print("Réponse Multi-Query")
print("------------------")
print(multi_query_answer)

---
## 2. Recherche HyDE

La question utilisateur et les passages de réponse n'utilisent pas toujours le même vocabulaire : une requête en langage naturel sera plus proche, sémantiquement, d'un texte de réponse que d'une autre question.

HyDE exploite cette asymétrie : on demande au LLM d'écrire un **document hypothétique**, un court passage factuel qui répondrait à la requête, puis on utilise ce texte comme vecteur de recherche à la place de la question brute.

### 2.1. Générer le document hypothétique

In [ ]:
hyde_system_prompt = """
Rédige un passage hypothétique pour la recherche HyDE.
Style: neutre, factuel, dense en information.
Inclure les entités/termes/synonymes probables de la requête pour améliorer la recherche.
Ne pas ajouter de méta-commentaire, de puces ni de JSON.
"""

hyde_result = await run_llm(
    LLMRequest(system_prompt=hyde_system_prompt, user_prompt=user_query)
)
hyde_text = hyde_result.output

### 2.2. Inspecter le document hypothétique

Ce texte sert de requête de recherche.
Vérifier qu'il ressemble à une vraie page de guide : lieux précis, détails concrets, style informatif.

In [ ]:
print("Document hypothétique généré par le LLM :")
print("=" * 60)
print(hyde_text)
print("=" * 60)
print(f"\nLongueur : {len(hyde_text)} caractères")

### 2.3. Récupérer les chunks avec le document hypothétique

In [ ]:
hyde_chunks = rag_assistant.search(query=hyde_text, top_k=HYDE_TOP_K)

### 2.4. Inspecter les chunks récupérés

In [ ]:
print(f"Total de chunks récupérés : {len(hyde_chunks)}")
print(f"Fichiers et volume :")
source_counts = {}
for chunk, _score in hyde_chunks:
    source = chunk.source
    if source in source_counts:
        source_counts[source] += 1
    else:
        source_counts[source] = 1
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")
print("\n"*5)

for rank, (chunk, score) in enumerate(hyde_chunks, start=1):
    print(f"#{rank} | score={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(f"  {chunk.text}")
    print("\n"*5)

### 2.5. Générer la réponse ancrée

In [ ]:
hyde_context = "\n\n".join(
    [f"[{chunk.source} - chunk {chunk.chunk_id}]\n{chunk.text}" for chunk, _score in hyde_chunks]
)
hyde_grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{hyde_context}"

hyde_run_result = await run_llm(
    LLMRequest(system_prompt=hyde_grounded_system_prompt, user_prompt=user_query)
)
hyde_answer = hyde_run_result.output

print("Réponse HyDE")
print("-----------")
print(hyde_answer)

---
## 3. Reranking

Le score vectoriel trie les chunks par proximité sémantique, mais ne capte pas toujours la pertinence métier : un chunk peut être proche en surface sans apporter d'information utile.

Le **reranking** confie ce reclassement à un LLM : on lui soumet tous les candidats et on lui demande de les ordonner selon leur utilité réelle pour la requête.

Comme pour le Multi-Query, on utilise `run_llm_structured` pour garantir un JSON valide directement, sans avoir à retirer soi-même des balises Markdown (` ```json `) autour de la réponse.

### 3.1. Fusionner et dédupliquer les candidats

In [ ]:
rerank_candidates = rag_deduplicate_and_sort_chunks(multi_query_chunks + hyde_chunks)

print(f"Chunks bruts Multi-Query : {len(multi_query_chunks)}")
print(f"Chunks HyDE              : {len(hyde_chunks)}")
print(f"Après fusion + déduplication : {len(rerank_candidates)}\n")
print("\n"*5)

print("Classement des candidats par score vectoriel (avant reranking LLM) :")
for index, (chunk, score) in enumerate(rerank_candidates, start=1):
    print(f"#{index} | score={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(f"  {chunk.text}")
    print("\n"*5)

### 3.2. Reclasser les candidats avec le LLM

Le LLM note chaque candidat par rapport à la requête utilisateur.
Cette étape coûte plus cher que la recherche vectorielle, mais elle peut remonter des chunks vraiment utiles et rétrograder les faux positifs.

In [ ]:
candidate_lines = []
for index, (chunk, _score) in enumerate(rerank_candidates, start=1):
    preview = chunk.text.replace("\n", " ")[:500]
    candidate_lines.append(f"ID={index} | source={chunk.source} | chunk_id={chunk.chunk_id} | text={preview}")

class RerankResult(BaseModel):
    ranking: list[int]  # IDs des candidats, du plus au moins pertinent


rerank_system_prompt = """
Tu es un module de reclassement de résultats de recherche. Note chaque passage selon sa capacité à répondre à la requête utilisateur.

À classer plus haut :
- Passages contenant des faits précis (prix, adresses, horaires, recommandations nommées)
- Passages alignés avec les contraintes utilisateur (budget, durée, préférences)

À classer plus bas :
- Sommaires, crédits éditoriaux, textes génériques
- Passages remontés par simple chevauchement lexical sans utilité réelle
"""

rerank_user_prompt = (
    f"Requête utilisateur:\n{user_query}\n\n"
    f"Sélectionne exactement {RERANK_FINAL_K} chunks parmi ces candidats, du plus pertinent au moins pertinent.\n\n"
    + "\n".join(candidate_lines)
)

rerank_result = await run_llm_structured(
    LLMRequest(system_prompt=rerank_system_prompt, user_prompt=rerank_user_prompt),
    response_schema=RerankResult,
)

ranking = json.loads(rerank_result.output)["ranking"]

### 3.3. Afficher le classement obtenu

In [ ]:
print(f"Reranking LLM : {ranking}\n")

### 3.4. Inspecter les résultats du reranking

Le tableau compare le nouveau rang (LLM) et l'ancien rang (vecteur).
Concrètement, ce delta montre la valeur ajoutée du reranking.

In [ ]:
reranked_chunks: list[tuple[RAGChunk, float, int]] = []
for new_rank, candidate_id in enumerate(ranking, start=1):
    idx = int(candidate_id) - 1
    if idx < 0 or idx >= len(rerank_candidates):
        print(f"Avertissement: ID hors plage ignoré {candidate_id} (taille du pool: {len(rerank_candidates)})")
        continue
    chunk, score = rerank_candidates[idx]
    reranked_chunks.append((chunk, score, int(candidate_id)))

print(f"IDs retournés par le LLM : {ranking}")
print(f"Pool candidat          : {len(rerank_candidates)} chunks")
print(f"Chunks valides conservés: {len(reranked_chunks)}\n")

print(f"{'New':>3} | {'Old':>3} | {'Vector':>6} | {'Source':<30} | Preview")
print("-" * 100)
for new_rank, (chunk, score, old_rank) in enumerate(reranked_chunks, start=1):
    vec_score = score
    preview = chunk.text.replace("\n", " ")[:60]
    direction = "↑" if new_rank < old_rank else ("↓" if new_rank > old_rank else "=")
    print(f"#{new_rank:>2} | #{old_rank:>2} {direction} | {vec_score:.4f} | {chunk.source:<30} | {preview}")

### 3.5. Générer la réponse ancrée

In [ ]:
rerank_context = "\n\n".join(
    [f"[{chunk.source} - chunk {chunk.chunk_id}]\n{chunk.text}" for chunk, _score, _old_rank in reranked_chunks]
)
rerank_grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{rerank_context}"

rerank_run_result = await run_llm(
    LLMRequest(system_prompt=rerank_grounded_system_prompt, user_prompt=user_query)
)
rerank_answer = rerank_run_result.output

print("Réponse après reranking")
print("----------------")
print(rerank_answer)


Les stratégies de recherche de ce notebook (Multi-Query, HyDE, Reranking) n'ont pas besoin d'être déplacées dans `shared/`.

Quand nous coderons un agent, on pourra créer un outil RAG, dans ce cas vous pourrez réutiliser les méthodes de retrieval avancée si vous le souhaitez !